In [1]:
!pip install yadisk

In [ ]:
import os
import yadisk
from tqdm.notebook import tqdm



TOKEN = "y0__xCE2dkeGKCzPCCUxrPaFdi_L_MOMvDW8BHwXyhrWt8KW_Bj"
DESTINATION_ROOT = "/Uploaded_Checkpoints"

paths_to_upload = [
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260107_110217/checkpoint-160000",
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260107_110217/live_config.json",
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260107_110217/training_config.json"
    
    
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260107_110217',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251221_112940',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251224_185822',
    '/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906',
    
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028",
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/checkpoint-200000",
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/live_config.json",
    # "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/training_config.json"
]

# Увеличиваем таймаут, так как файлы огромные
y = yadisk.Client(token=TOKEN)

class ProgressReader:
    def __init__(self, filename, pbar):
        self.file = open(filename, 'rb')
        self.pbar = pbar

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

    def read(self, size=-1):
        data = self.file.read(size)
        current_position = self.file.tell()
        
        self.pbar.n = current_position
        self.pbar.update(0) 
        
        return data

    def close(self):
        self.file.close()

    def __getattr__(self, attr):
        return getattr(self.file, attr)

def ensure_remote_dir(path):
    parts = path.strip("/").split("/")
    current_path = ""
    for part in parts:
        current_path = f"{current_path}/{part}"
        if not y.exists(current_path):
            try:
                y.mkdir(current_path)
            except yadisk.exceptions.PathExistsError:
                pass

def upload_file(local_path, remote_path):
    file_size = os.path.getsize(local_path)
    ensure_remote_dir(os.path.dirname(remote_path))
    
    # Проверяем, может файл уже залит такого же размера (докачка "вручную")
    if y.exists(remote_path):
        remote_meta = y.get_meta(remote_path)
        if remote_meta.size == file_size:
            print(f"Skipping {local_path} (already exists and size matches)")
            return

    with tqdm(total=file_size, unit="B", unit_scale=True, desc=local_path, mininterval=1.0) as pbar:
        try:
            with ProgressReader(local_path, pbar) as f:
                y.upload(f, remote_path, overwrite=False, timeout=(10, 60), skip_exists=True)
        except Exception as e:
            print(f"\n❌ Error uploading {local_path}: {e}")

def process_path(local_path, base_remote_path):
    base_name = os.path.basename(local_path)
    
    remote_dest = f"{base_remote_path}/{base_name}"

    if os.path.isfile(local_path):
        upload_file(local_path, remote_dest)
    
    elif os.path.isdir(local_path):
        for root, _, files in os.walk(local_path):
            for file in files:
                full_local = os.path.join(root, file)
                rel_path = os.path.relpath(full_local, local_path)
                full_remote = f"{remote_dest}/{rel_path}"
                upload_file(full_local, full_remote)

if not y.check_token():
    print("Invalid Token")
else:
    print("Token OK. Starting upload...")
    ensure_remote_dir(DESTINATION_ROOT)
    for path in paths_to_upload:
        process_path(path, DESTINATION_ROOT)


Token OK. Starting upload...
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/live_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/training_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/checkpoint-40000/model.pt (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/live_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/training_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/checkpoint-200000/model.pt (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/c

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/checkpoint-120000/model.pt: …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/live_config.json:   0%|     …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/training_config.json:   0%| …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251228_230459/checkpoint-160000/model.pt: …

...

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906/checkpoint-160000/model.pt: …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906/checkpoint-240000/model.pt: …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906/checkpoint-320000/model.pt: …


❌ Error uploading /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906/checkpoint-320000/model.pt: Ресурс "disk:/Uploaded_Checkpoints/run_20260102_155906/checkpoint-320000/model.pt" уже существует. | Error description: Resource "disk:/Uploaded_Checkpoints/run_20260102_155906/checkpoint-320000/model.pt" already exists. | Error code: DiskResourceAlreadyExistsError | Status code: 409


/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20260102_155906/checkpoint-400000/model.pt: …

In [1]:
from tqdm.auto import tqdm

In [2]:
for i in tqdm(range(10000000)):
    a = 1
    a += 1

  0%|          | 0/10000000 [00:00<?, ?it/s]